# Step 12: Retrain Model 2's chemistry-pretrained base checkpoint with a fixed vocabulary (Colab GPU)

`08_train_conditions_model_reactiont5base.ipynb`'s run (`model2_conditions_reactiont5base`)
finished with a healthy training `eval_loss` (0.14) but **0% valid generation on all 2285
test records** (RESULTS.md, "Спроба: хімічно-передтренований базовий чекпоінт").

**Root cause (confirmed by direct tokenizer testing):** `sagawa/ReactionT5v2-retrosynthesis`'s
268-token vocabulary was built purely from SMILES chemistry notation. It has no tokens for
most JSON punctuation (`{`, `}`, `"`, `:`, `,`) or most English letters -- only the ones that
double as SMILES atom symbols survive. Since this task's targets are JSON strings with
English filler text (`"not specified"`), most training targets were silently collapsing into
an unrecoverable `<unk>`-riddled mess. Teacher-forced training loss looked fine because the
model was correctly learning to reproduce the *corrupted* (but self-consistent) targets --
actual autoregressive generation had no valid completions to produce.

**Fix (`scripts/train_conditions_model.py`, `ensure_full_char_coverage`):** before training,
scan the full train+val corpus, `tokenizer.add_tokens()` any character that maps to `<unk>`,
and `model.resize_token_embeddings()` to match. Verified locally: 65 new single-character
tokens added (vocab 268→303), 0 remaining `<unk>` on the exact JSON target string that
previously produced 24 `<unk>` tokens out of 56. This is otherwise the same run as step 8:
same data (`data/v2_ord_train/conditions_{train,val,test}.jsonl`, 41,139/2,285/2,285), same
`lr=5e-5`, same ~170 min time budget.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Reclaim Drive quota (optional, run once per session):** the training script no
longer rotates/deletes checkpoints directly on Drive at all -- Trainer checkpoints on
local Colab disk and only ever *overwrites* one fixed Drive folder
(`{output_dir}/latest_checkpoint`), which sidesteps Drive's Trash-on-delete behavior
entirely. Still useful once, to clear out anything trashed by earlier runs. First run
prompts an auth popup.

**Warning:** this empties Trash for your **entire** Google Drive account, not just
this project's files -- anything else you'd trashed elsewhere and might still want
to recover will be gone permanently too. Skip this cell if that matters to you.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
build("drive", "v3").files().emptyTrash().execute()
print("Drive Trash emptied.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` is gitignored (large, derived) -- regenerate it deterministically
here (fixed seed, excludes the committed `data/v2_ord_eval_targets.json` by
construction). Only needs to run once per Colab session.

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/conditions_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42

**Cross-account resume:** Colab sessions may run under different Google accounts/Drives
each time, so a previous session's `{output_dir}/latest_checkpoint` isn't reliably
visible automatically. Two ways to hand a checkpoint to the next session:

- **Recommended (faster for large checkpoints):** on drive.google.com (in whichever
  account is mounted *this* session), drag-and-drop the checkpoint folder anywhere
  under My Drive -- the Drive website handles large-folder uploads more reliably than
  a browser file picker. Then type its path into `resume_from_checkpoint_path` in the
  next cell -- skip the upload-widget cell entirely.
- **Alternative:** the upload-widget cell below (goes through the browser, slower for
  large folders).

Skip both for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /content/drive/MyDrive/retro-planner-checkpoints/model2_conditions/final
# Leave blank if you're using the upload-widget cell below instead, or if this is a first run.

In [ ]:
import os
import shutil
import zipfile

from google.colab import files

uploaded = files.upload()  # skip this cell (don't run it) if you set resume_from_checkpoint_path above instead
if uploaded:
    zip_name = next(iter(uploaded))
    extract_dir = "/content/resume_from"
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_dir)
    entries = os.listdir(extract_dir)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        extract_dir = os.path.join(extract_dir, entries[0])
    resume_from_checkpoint_path = extract_dir
    print(f"Will resume from: {resume_from_checkpoint_path}")
    print("Contents:", os.listdir(resume_from_checkpoint_path))

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model2_conditions_reactiont5base_vocabfix"  # @param {type:"string"}
time_budget_minutes = 170  # @param {type:"number"}
base_model = "sagawa/ReactionT5v2-retrosynthesis"  # @param {type:"string"}


In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!python scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --learning-rate 5e-5 \
    --per-device-train-batch-size 16 \
    --per-device-eval-batch-size 16 \
    --gradient-accumulation-steps 2 \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is redirected to `train.log` in `output_dir` (on Drive) instead of
printing here, to avoid the notebook's output growing large enough to make the browser
tab unresponsive on a long run. Open `train.log` in Google Drive's own web preview to
check progress -- that works independently of the Colab kernel, which stays busy
(blocked) running the cell above.

**To continue in a later session** (possibly under a different Google account): download
`{output_dir}/final` or a `checkpoint-N` from `{output_dir}/latest_checkpoint` as a zip,
then upload it in the "Cross-account resume" cell above next time. Same-account
reconnects auto-resume from `{output_dir}/latest_checkpoint` without needing an upload.
Once finished, evaluate against `data/v2_ord_train/conditions_test.jsonl` (held out, never
used in training) -- same file as `06_train_conditions_model.ipynb`'s t5-small run, so the
two are directly comparable.